# Yorùbá OCR Research — Google Colab Pipeline

**Runtime:** GPU (T4 or better) → Runtime → Change runtime type → T4 GPU

**Steps:**
1. Mount Drive & clone repo
2. Install dependencies
3. Clone PaddleOCR
4. Run Phase 01-03 (Data → Config)
5. Run Phase 04 (Fine-tune on GPU)
6. Run Phase 05-09 (Eval + Baselines + Results)
7. Backup outputs to Drive

## Step 1 — Mount Drive & Set Up Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'
REPO_DIR   = f'{DRIVE_ROOT}/yoruba_ocr_research'

print(f'Drive mounted. Repo will live at: {REPO_DIR}')

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/sam4rano/yoruba_ocr_research.git "{REPO_DIR}"
else:
    print('Repo already exists — pulling latest changes...')
    !git -C "{REPO_DIR}" pull

%cd "{REPO_DIR}"
!pwd

## Step 2 — Install Python Dependencies

In [ ]:
# Install PaddlePaddle GPU (Colab Linux x86 with CUDA)
!python -m pip install paddlepaddle-gpu -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html -q

In [ ]:
# Install all other project requirements (skip paddlepaddle line — already installed above)
!grep -v '^paddlepaddle' requirements.txt | grep -v '^#' | grep -v '^$' > /tmp/reqs_no_paddle.txt
!python -m pip install -r /tmp/reqs_no_paddle.txt -q
print('All dependencies installed.')

In [ ]:
# Verify paddle is importable and sees the GPU
import paddle
print('PaddlePaddle version:', paddle.__version__)
print('GPU available:', paddle.device.is_compiled_with_cuda())
print('Device count :', paddle.device.cuda.device_count())

## Step 3 — Clone PaddleOCR Repo

In [ ]:
import os

if not os.path.isdir('PaddleOCR'):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git PaddleOCR
else:
    print('PaddleOCR already cloned — skipping.')

# Install PaddleOCR's own deps (lightweight, most already satisfied)
!python -m pip install -r PaddleOCR/requirements.txt -q
print('PaddleOCR ready.')

## Step 4 — Data Consolidation, Analysis & Config (Phases 01-03)

In [ ]:
import os

# The processed dataset ships with the repo (scripts/01 already ran locally).
# If data/processed is missing or empty, re-run consolidation from raw.
processed_labels = 'data/processed/labels/train.txt'

if os.path.isfile(processed_labels):
    import subprocess
    n = int(subprocess.check_output(['wc', '-l', processed_labels]).split()[0])
    print(f'data/processed already present — {n} train lines. Skipping Phase 01.')
else:
    print('data/processed missing — running Phase 01 (consolidation)...')
    !bash scripts/shell/phase_01_consolidate.sh

In [ ]:
# Phase 02 — Dataset analysis
!bash scripts/shell/phase_02_analyze.sh

# Phase 03 — Generate PaddleOCR YAML config (GPU mode)
!CONFIG_FORCE_GPU=1 bash scripts/shell/phase_03_config.sh

print('Phases 02 and 03 complete.')

In [ ]:
# Sanity check: show key fields from the generated config
import yaml
with open('configs/paddleocr_yoruba_rec.yml') as f:
    cfg = yaml.safe_load(f)

g = cfg.get('Global', {})
print('use_gpu  :', g.get('use_gpu'))
print('char_dict:', g.get('character_dict_path'))
print('epochs   :', g.get('epoch_num'))
print('save_dir :', g.get('save_model_dir'))

## Step 5 — Fine-Tune PaddleOCR on GPU (Phase 04)

In [ ]:
# Run fine-tuning. GPU 0 is used by default.
# Set TRAIN_EPOCHS_OVERRIDE to a smaller number (e.g. 10) for a quick smoke-test.

import os
os.environ['EVAL_USE_GPU']  = '1'
os.environ['CONFIG_FORCE_GPU'] = '1'
# os.environ['TRAIN_EPOCHS_OVERRIDE'] = '10'  # Uncomment for a quick smoke-test

!bash scripts/shell/phase_04_train.sh

## Step 6 — Evaluate Baselines & Fine-Tuned Model (Phases 05-08)

In [ ]:
# Phase 05: Evaluate PaddleOCR (pretrained + fine-tuned)
!EVAL_USE_GPU=1 bash scripts/shell/phase_05_eval_paddle.sh

In [ ]:
# Install Tesseract system binary + Yoruba language pack
!apt-get install -qq tesseract-ocr tesseract-ocr-yor 2>/dev/null

# Phase 06: Tesseract baseline
!bash scripts/shell/phase_06_tesseract.sh

In [ ]:
# Phase 07: Qwen 2.5-VL zero-shot baseline
# Note: needs ~16GB VRAM. Set QWEN_SKIP=1 to skip on small GPUs.
import os
# os.environ['QWEN_SKIP'] = '1'  # Uncomment to skip Qwen
!bash scripts/shell/phase_07_qwen.sh

In [ ]:
# Phase 08: Ablation study
!bash scripts/shell/phase_08_ablation.sh

## Step 7 — Compile Results (Phase 09)

In [ ]:
!bash scripts/shell/phase_09_compile.sh

# Show the final metrics summary table
import pandas as pd
df = pd.read_csv('results/tables/metrics_summary.csv')
print(df.to_string(index=False))

## Step 8 — Backup Results to Google Drive (Phase 99)

In [ ]:
import os
os.environ['DRIVE_BACKUP_ROOT'] = f'{DRIVE_ROOT}/yoruba_ocr_backups'
os.environ['BACKUP_EXPERIMENTS'] = '1'  # also backup model checkpoints

!bash scripts/shell/phase_99_backup.sh
print('Backup complete. Check your Google Drive → yoruba_ocr_backups/')